# Exercise 06 — Design Patterns

This module covers five foundational patterns from [refactoring.guru](https://refactoring.guru/design-patterns/catalog), grouped by category:

| Pattern | Category | Core Idea |
|---------|----------|-----------|
| **Singleton** | Creational | Only one instance of a class ever exists |
| **Factory Method** | Creational | Let subclasses/a factory decide which class to instantiate |
| **Strategy** | Behavioral | Swap algorithms/behaviors at runtime |
| **Observer** | Behavioral | Notify many objects when one object changes |
| **Decorator** | Structural | Wrap objects to add behavior without subclassing |

> **Note:** Strategy and Observer appeared briefly in Exercise 05. Here they get a deeper treatment with more realistic scenarios.

---
## Pattern 1 — Singleton (Creational)

### Concept

A Singleton ensures a class has **exactly one instance** and provides a global access point to it.

| Term | What it means |
|------|---------------|
| `__new__` | Called before `__init__`; controls object creation |
| `_instance` | Class-level variable holding the single instance |
| Global access point | `MyClass()` always returns the same object |

```python
class Registry:
    _instance = None

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance

r1 = Registry()
r2 = Registry()
assert r1 is r2  # same object every time
```

**When to use:** logging, config stores, connection pools, caches — anywhere you need one shared state.

---
## Task 1: Implement `AppConfig`

A global key-value configuration store. Any code that calls `AppConfig()` must get the **same** instance.

```python
cfg1 = AppConfig()
cfg2 = AppConfig()
assert cfg1 is cfg2                     # same object

cfg1.set("debug", True)
cfg2.get("debug")                       # True  (shared state)

cfg1.set("timeout", 30)
cfg1.get("timeout")                     # 30
cfg1.get("missing", default="n/a")      # "n/a"
cfg1.get("missing")                     # raises KeyError
```

**Rules:**
1. Use `__new__` to enforce a single instance
2. `set(key, value)` stores the pair
3. `get(key)` returns the value or raises `KeyError` if missing
4. `get(key, default=...)` returns `default` instead of raising
5. `all()` returns a copy of the entire config dict

In [ ]:
# YOUR CODE HERE

class AppConfig:
    pass  # Replace with your implementation

In [ ]:
# TEST — run this cell to check your solution

# Reset singleton for test isolation
AppConfig._instance = None

cfg1 = AppConfig()
cfg2 = AppConfig()
assert cfg1 is cfg2, "Must be the same instance"

# Shared state
cfg1.set("debug", True)
assert cfg2.get("debug") is True

cfg1.set("timeout", 30)
assert cfg1.get("timeout") == 30

# Default value
assert cfg1.get("missing", default="n/a") == "n/a"

# KeyError when no default
try:
    cfg1.get("missing")
    assert False, "Should have raised KeyError"
except KeyError:
    pass

# all() returns a copy (mutating it should not affect the store)
snapshot = cfg1.all()
assert snapshot["debug"] is True
snapshot["debug"] = False
assert cfg1.get("debug") is True  # original unchanged

print("✓ All Singleton tests passed!")

---
## Pattern 2 — Factory Method (Creational)

### Concept

Define an interface for creating an object, but let a **factory function or subclass** decide which class to instantiate. The caller never uses `ClassName()` directly.

| Term | What it means |
|------|---------------|
| **Product** | The abstract interface all created objects share |
| **Concrete Product** | The actual class being created |
| **Factory** | The function/class that produces products |
| **Factory Method** | The method/function that decides what to create |

```python
from abc import ABC, abstractmethod

class Transport(ABC):
    @abstractmethod
    def deliver(self) -> str: ...

class Truck(Transport):
    def deliver(self): return "Deliver by land"

class Ship(Transport):
    def deliver(self): return "Deliver by sea"

def create_transport(mode: str) -> Transport:
    """Factory method — caller never touches Truck/Ship directly."""
    if mode == "land": return Truck()
    if mode == "sea":  return Ship()
    raise ValueError(f"Unknown mode: {mode}")

t = create_transport("land")
print(t.deliver())  # "Deliver by land"
```

**When to use:** when the exact type to create is only known at runtime, or when you want to decouple creation from usage.

---
## Task 2: Implement a Logger Factory

Build a logging system where different backends can be created via a factory.

```python
# All loggers share the same interface
logger = create_logger("console")
logger.log("hello")         # "[CONSOLE] hello"

logger = create_logger("file", path="app.log")
logger.log("hello")         # "[FILE:app.log] hello"

logger = create_logger("json")
logger.log("hello")         # '{"level": "INFO", "message": "hello"}'

create_logger("unknown")    # raises ValueError
```

**Classes to implement:**
```
Logger (ABC)
  ├── ConsoleLogger    → "[CONSOLE] {msg}"
  ├── FileLogger       → "[FILE:{path}] {msg}"  (path passed at creation)
  └── JsonLogger       → '{"level": "INFO", "message": "{msg}"}'

create_logger(backend: str, **kwargs) -> Logger
```

**Rules:**
1. `Logger` is abstract with one abstract method: `log(message: str) -> str`
2. Each concrete logger returns (and prints) the formatted string
3. `create_logger` raises `ValueError` for unknown backends
4. `FileLogger` receives `path` as a keyword argument

In [ ]:
import json
from abc import ABC, abstractmethod

# YOUR CODE HERE

class Logger(ABC):
    @abstractmethod
    def log(self, message: str) -> str:
        pass


class ConsoleLogger(Logger):
    pass


class FileLogger(Logger):
    pass


class JsonLogger(Logger):
    pass


def create_logger(backend: str, **kwargs) -> Logger:
    pass

In [ ]:
# TEST — run this cell to check your solution

# ConsoleLogger
logger = create_logger("console")
assert isinstance(logger, Logger)
assert logger.log("hello") == "[CONSOLE] hello"

# FileLogger
logger = create_logger("file", path="app.log")
assert isinstance(logger, Logger)
assert logger.log("hello") == "[FILE:app.log] hello"

# JsonLogger
logger = create_logger("json")
assert isinstance(logger, Logger)
result = logger.log("hello")
parsed = json.loads(result)
assert parsed["level"] == "INFO"
assert parsed["message"] == "hello"

# Unknown backend
try:
    create_logger("unknown")
    assert False, "Should have raised ValueError"
except ValueError:
    pass

# Abstract class cannot be instantiated
try:
    Logger()
    assert False, "Should have raised TypeError"
except TypeError:
    pass

# All loggers share the same interface (polymorphism)
loggers = [
    create_logger("console"),
    create_logger("file", path="out.log"),
    create_logger("json"),
]
for lg in loggers:
    result = lg.log("test")
    assert isinstance(result, str)

print("✓ All Factory Method tests passed!")

---
## Pattern 3 — Strategy (Behavioral)

### Concept

Define a family of algorithms, encapsulate each one, and make them **interchangeable at runtime**. The context object delegates work to the strategy object.

| Term | What it means |
|------|---------------|
| **Context** | The object that uses a strategy (`ShoppingCart`, `Sorter`) |
| **Strategy** | The abstract interface for the algorithm |
| **Concrete Strategy** | A specific algorithm implementation |
| **Delegation** | Context calls `strategy.execute()`, not its own logic |

```python
from abc import ABC, abstractmethod

class SortStrategy(ABC):
    @abstractmethod
    def sort(self, data: list) -> list: ...

class AscendingSort(SortStrategy):
    def sort(self, data): return sorted(data)

class DescendingSort(SortStrategy):
    def sort(self, data): return sorted(data, reverse=True)

class DataProcessor:
    def __init__(self, strategy: SortStrategy):
        self.strategy = strategy

    def process(self, data): return self.strategy.sort(data)

p = DataProcessor(AscendingSort())
p.process([3, 1, 2])   # [1, 2, 3]
p.strategy = DescendingSort()  # swap at runtime
p.process([3, 1, 2])   # [3, 2, 1]
```

**vs Exercise 05:** Exercise 05 showed the basic idea. Here the context carries richer state (cart items, totals) and strategies implement real business logic.

---
## Task 3: Implement a Discount System

A `ShoppingCart` applies a discount strategy to its total.

```python
cart = ShoppingCart()
cart.add_item("apple",  1.50)
cart.add_item("banana", 2.00)
cart.add_item("cherry", 3.50)  # subtotal = 7.00

# No discount
cart.set_strategy(NoDiscount())
cart.total()   # 7.00

# Flat percentage off
cart.set_strategy(PercentageDiscount(10))  # 10% off
cart.total()   # 6.30

# Buy-one-get-one: cheapest item free
cart.set_strategy(BuyOneGetOneDiscount())
cart.total()   # 5.50  (cheapest item = 1.50 is free)

# Spend threshold: £5 off when subtotal >= £50
cart.set_strategy(ThresholdDiscount(threshold=50, saving=5))
cart.total()   # 7.00  (threshold not met)
```

**Classes to implement:**
```
DiscountStrategy (ABC)
  ├── NoDiscount
  ├── PercentageDiscount(percent)
  ├── BuyOneGetOneDiscount   ← cheapest item is free
  └── ThresholdDiscount(threshold, saving)

ShoppingCart
  .add_item(name, price)
  .set_strategy(strategy)
  .total() -> float
```

**Rules:**
1. `DiscountStrategy` is abstract with one method: `apply(prices: list[float]) -> float` (returns final total)
2. `BuyOneGetOneDiscount` makes the single cheapest item free
3. `ThresholdDiscount` only applies if subtotal >= threshold
4. Default strategy is `NoDiscount` (cart works without calling `set_strategy`)

In [ ]:
from abc import ABC, abstractmethod

# YOUR CODE HERE

class DiscountStrategy(ABC):
    @abstractmethod
    def apply(self, prices: list) -> float:
        pass


class NoDiscount(DiscountStrategy):
    pass


class PercentageDiscount(DiscountStrategy):
    pass


class BuyOneGetOneDiscount(DiscountStrategy):
    pass


class ThresholdDiscount(DiscountStrategy):
    pass


class ShoppingCart:
    pass

In [ ]:
# TEST — run this cell to check your solution

cart = ShoppingCart()
cart.add_item("apple",  1.50)
cart.add_item("banana", 2.00)
cart.add_item("cherry", 3.50)  # subtotal = 7.00

# Default / NoDiscount
assert abs(cart.total() - 7.00) < 1e-9

cart.set_strategy(NoDiscount())
assert abs(cart.total() - 7.00) < 1e-9

# PercentageDiscount — 10% off
cart.set_strategy(PercentageDiscount(10))
assert abs(cart.total() - 6.30) < 1e-9

# BuyOneGetOneDiscount — cheapest item (1.50) free
cart.set_strategy(BuyOneGetOneDiscount())
assert abs(cart.total() - 5.50) < 1e-9

# ThresholdDiscount — threshold not met
cart.set_strategy(ThresholdDiscount(threshold=50, saving=5))
assert abs(cart.total() - 7.00) < 1e-9

# ThresholdDiscount — threshold met
big_cart = ShoppingCart()
big_cart.add_item("item", 60.00)
big_cart.set_strategy(ThresholdDiscount(threshold=50, saving=5))
assert abs(big_cart.total() - 55.00) < 1e-9

# Strategy swap at runtime
cart.set_strategy(NoDiscount())
assert abs(cart.total() - 7.00) < 1e-9
cart.set_strategy(PercentageDiscount(50))
assert abs(cart.total() - 3.50) < 1e-9

print("✓ All Strategy tests passed!")

---
## Pattern 4 — Observer (Behavioral)

### Concept

Define a **one-to-many dependency** so that when one object (the **subject**) changes state, all its dependents (**observers**) are notified automatically.

| Term | What it means |
|------|---------------|
| **Subject / Publisher** | Holds state; notifies observers on change |
| **Observer / Subscriber** | Reacts to state changes via `update()` |
| `subscribe(obs)` | Register an observer |
| `unsubscribe(obs)` | Remove an observer |
| `notify()` | Calls `update()` on all registered observers |

```python
from abc import ABC, abstractmethod

class Observer(ABC):
    @abstractmethod
    def update(self, event: str, data): ...

class Subject:
    def __init__(self):
        self._observers = []

    def subscribe(self, obs): self._observers.append(obs)
    def unsubscribe(self, obs): self._observers.remove(obs)
    def notify(self, event, data):
        for obs in self._observers:
            obs.update(event, data)
```

**vs Exercise 05 EventBus:** EventBus uses string event names (pub/sub style). Classic Observer is tighter-coupled: observers register directly on the subject and receive structured data.

---
## Task 4: Implement a Stock Ticker

A `StockMarket` (subject) broadcasts price changes to multiple observers.

```python
market = StockMarket()

logger   = PriceLogger()      # logs every change
alert    = PriceAlert(threshold=150)  # fires when price exceeds threshold
portfolio = Portfolio("AAPL", shares=10)  # tracks portfolio value

market.subscribe(logger)
market.subscribe(alert)
market.subscribe(portfolio)

market.set_price("AAPL", 140)
# logger records: "AAPL: 140"
# alert: silent (140 < 150)
# portfolio value: 1400

market.set_price("AAPL", 160)
# logger records: "AAPL: 160"
# alert fires: "ALERT: AAPL exceeded 150 at 160"
# portfolio value: 1600

market.unsubscribe(logger)
market.set_price("AAPL", 170)  # logger does NOT record this
```

**Classes to implement:**
```
StockObserver (ABC)
  .update(ticker: str, price: float)

StockMarket (Subject)
  .subscribe(observer)
  .unsubscribe(observer)
  .set_price(ticker, price)   ← triggers notify

PriceLogger    → stores (ticker, price) tuples in .history list
PriceAlert     → stores alert strings in .alerts list when price > threshold
Portfolio      → tracks .value = shares * latest_price for its ticker
```

In [ ]:
from abc import ABC, abstractmethod

# YOUR CODE HERE

class StockObserver(ABC):
    @abstractmethod
    def update(self, ticker: str, price: float):
        pass


class StockMarket:
    pass


class PriceLogger(StockObserver):
    pass


class PriceAlert(StockObserver):
    pass


class Portfolio(StockObserver):
    pass

In [ ]:
# TEST — run this cell to check your solution

market    = StockMarket()
logger    = PriceLogger()
alert     = PriceAlert(threshold=150)
portfolio = Portfolio("AAPL", shares=10)

market.subscribe(logger)
market.subscribe(alert)
market.subscribe(portfolio)

# First price update
market.set_price("AAPL", 140)
assert ("AAPL", 140) in logger.history
assert len(alert.alerts) == 0          # below threshold
assert abs(portfolio.value - 1400) < 1e-9

# Second price update — crosses threshold
market.set_price("AAPL", 160)
assert ("AAPL", 160) in logger.history
assert len(alert.alerts) == 1
assert "AAPL" in alert.alerts[0] and "160" in alert.alerts[0]
assert abs(portfolio.value - 1600) < 1e-9

# Unsubscribe logger; portfolio still updates
market.unsubscribe(logger)
market.set_price("AAPL", 170)
assert len(logger.history) == 2        # logger saw only the first two
assert abs(portfolio.value - 1700) < 1e-9

# Portfolio only cares about its ticker
market.subscribe(logger)
market.set_price("GOOG", 200)          # different ticker
assert abs(portfolio.value - 1700) < 1e-9  # AAPL portfolio unchanged

# Multiple alerts accumulate
market.set_price("AAPL", 180)
assert len(alert.alerts) == 2

print("✓ All Observer tests passed!")

---
## Pattern 5 — Decorator (Structural)

### Concept

Attach **additional responsibilities to an object dynamically** by wrapping it in decorator objects that share the same interface. Avoids an explosion of subclasses.

| Term | What it means |
|------|---------------|
| **Component** | The base interface all objects share |
| **Concrete Component** | The base object being decorated |
| **Decorator** | Wraps a component; delegates + adds behavior |
| **Wrapping** | `self._component.method()` + extra logic |

```python
class TextComponent(ABC):
    @abstractmethod
    def render(self) -> str: ...

class PlainText(TextComponent):
    def __init__(self, text): self._text = text
    def render(self): return self._text

class UpperCaseDecorator(TextComponent):
    def __init__(self, component): self._component = component
    def render(self): return self._component.render().upper()

class ExclamationDecorator(TextComponent):
    def __init__(self, component): self._component = component
    def render(self): return self._component.render() + "!"

# Decorators stack — order matters
text = ExclamationDecorator(UpperCaseDecorator(PlainText("hello")))
text.render()  # "HELLO!"
```

**When to use:** adding optional features (logging, caching, auth, compression) without modifying the original class or creating a subclass for every combination.

---
## Task 5: Implement a Coffee Order Builder

The classic Decorator example. A base `Coffee` costs a fixed amount. Decorators add ingredients and cost.

```python
order = Espresso()                  # cost: 2.00, desc: "Espresso"
order = MilkDecorator(order)        # cost: 2.50, desc: "Espresso, Milk"
order = SugarDecorator(order)       # cost: 2.75, desc: "Espresso, Milk, Sugar"
order = WhipDecorator(order)        # cost: 3.25, desc: "Espresso, Milk, Sugar, Whip"

order.cost()         # 3.25
order.description()  # "Espresso, Milk, Sugar, Whip"

# Can stack same decorator multiple times
double_milk = MilkDecorator(MilkDecorator(Espresso()))
double_milk.cost()          # 3.00
double_milk.description()   # "Espresso, Milk, Milk"
```

**Ingredient costs:**
| Item | Cost |
|------|------|
| Espresso (base) | 2.00 |
| Milk | +0.50 |
| Sugar | +0.25 |
| Whip | +0.75 |

**Classes to implement:**
```
Beverage (ABC)
  .cost() -> float
  .description() -> str

Espresso(Beverage)          ← concrete base

CondimentDecorator(Beverage)  ← abstract decorator base
  MilkDecorator
  SugarDecorator
  WhipDecorator
```

In [ ]:
from abc import ABC, abstractmethod

# YOUR CODE HERE

class Beverage(ABC):
    @abstractmethod
    def cost(self) -> float:
        pass

    @abstractmethod
    def description(self) -> str:
        pass


class Espresso(Beverage):
    pass


class CondimentDecorator(Beverage):
    """Abstract base for all decorators — stores reference to wrapped beverage."""
    pass


class MilkDecorator(CondimentDecorator):
    pass


class SugarDecorator(CondimentDecorator):
    pass


class WhipDecorator(CondimentDecorator):
    pass

In [ ]:
# TEST — run this cell to check your solution

# Base
order = Espresso()
assert abs(order.cost() - 2.00) < 1e-9
assert order.description() == "Espresso"

# Add Milk
order = MilkDecorator(order)
assert abs(order.cost() - 2.50) < 1e-9
assert order.description() == "Espresso, Milk"

# Add Sugar
order = SugarDecorator(order)
assert abs(order.cost() - 2.75) < 1e-9
assert order.description() == "Espresso, Milk, Sugar"

# Add Whip
order = WhipDecorator(order)
assert abs(order.cost() - 3.25) < 1e-9
assert order.description() == "Espresso, Milk, Sugar, Whip"

# All decorators implement Beverage (polymorphism)
assert isinstance(order, Beverage)

# Double milk
double_milk = MilkDecorator(MilkDecorator(Espresso()))
assert abs(double_milk.cost() - 3.00) < 1e-9
assert double_milk.description() == "Espresso, Milk, Milk"

# Different order of decorators
order2 = SugarDecorator(WhipDecorator(Espresso()))
assert abs(order2.cost() - 3.00) < 1e-9
assert order2.description() == "Espresso, Whip, Sugar"

print("✓ All Decorator tests passed!")

---
## Summary & Comparison

| Pattern | Who creates objects? | Who decides behavior? | Key signal in code |
|---------|---------------------|----------------------|-------------------|
| **Singleton** | The class controls its own creation | — | `__new__` + `_instance` |
| **Factory Method** | A factory function / method | — | `create_x(type)` returning ABC |
| **Strategy** | Caller | Swappable strategy object | `context.strategy = ...` |
| **Observer** | Caller | Subject notifies all observers | `subject.notify()` loop |
| **Decorator** | Caller stacks wrappers | Each wrapper adds behavior | `Wrapper(Wrapper(Base()))` |

### Common Interview Questions

1. **Singleton vs global variable** — What's the difference? (Answer: Singleton controls instantiation via `__new__`, lazy-initializes, and can be subclassed)
2. **Strategy vs if/elif** — When would you refactor conditionals to Strategy?
3. **Observer vs direct method calls** — What does loose coupling buy you?
4. **Decorator vs inheritance** — Why prefer composition here? (Answer: avoids class explosion, stacks at runtime)
5. **Factory vs direct construction** — When does hiding `ClassName()` add value?